# T07FakeMediaDetect - Full Training from Scratch (CNN + Hybrid)

**Notebook này dùng để:**
1.  Tải dataset CASIA 2.0.
2.  **Huấn luyện CNN từ đầu** (Phase 1) để nhận diện ảnh ELA.
3.  **Huấn luyện SVM** (Phase 2) kết hợp kết quả CNN và Luật Benford.
4.  Đánh giá và vẽ biểu đồ kết quả.

Dùng notebook này khi bạn **chưa có file model .h5**.

## 1. Setup & Dependencies

In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn opencv-python Pillow tensorflow kaggle

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, ImageChops, ImageEnhance
from tqdm.notebook import tqdm
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, precision_recall_curve
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

## 2. Download Dataset

In [ ]:
import os
from google.colab import files

if not os.path.exists('datasets'):
    os.makedirs('datasets')

# Upload kaggle.json if not exists
if not os.path.exists('kaggle.json'):
    print("Vui lòng upload file kaggle.json để tải dataset...")
    uploaded = files.upload()

if os.path.exists('kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d divg07/casia-20-image-tampering-detection-dataset -p datasets --unzip
    print("Download xong!")
else:
    print("Không tìm thấy kaggle.json. Bạn cần upload dataset thủ công vào thư mục 'datasets'.")

In [ ]:
def load_image_paths(base_path, sample_size=None):
    authentic_paths = []
    forged_paths = []
    
    print(f"Scanning: {base_path}")
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.tif')):
                path = os.path.join(root, file)
                if 'au' in root.lower() or 'original' in root.lower():
                    authentic_paths.append(path)
                elif 'tp' in root.lower() or 'forged' in root.lower() or 'tampered' in root.lower():
                    forged_paths.append(path)
    
    print(f"Found {len(authentic_paths)} Authentic, {len(forged_paths)} Forged")
    
    # Balancing
    min_len = min(len(authentic_paths), len(forged_paths))
    if sample_size and sample_size < min_len:
        min_len = sample_size
        
    authentic_paths = random.sample(authentic_paths, min_len)
    forged_paths = random.sample(forged_paths, min_len)
    
    paths = authentic_paths + forged_paths
    labels = [0] * len(authentic_paths) + [1] * len(forged_paths)
    
    df = pd.DataFrame({'path': paths, 'label': labels})
    df = df.sample(frac=1).reset_index(drop=True)
    return df

# Load Dataframe
df = load_image_paths('datasets', sample_size=2000) # 2000 mỗi loại = 4000 ảnh tổng
print(df.head())

## 3. Preprocessing Functions

In [ ]:
def convert_to_ela_image(path, quality=90):
    temp_filename = 'temp_ela.jpg'
    try:
        image = Image.open(path).convert('RGB')
        image.save(temp_filename, 'JPEG', quality=quality)
        resaved = Image.open(temp_filename)
        ela = ImageChops.difference(image, resaved)
        extrema = ela.getextrema()
        max_diff = max([ex[1] for ex in extrema])
        if max_diff == 0: max_diff = 1
        scale = 255.0 / max_diff
        return ImageEnhance.Brightness(ela).enhance(scale)
    except:
        return None

def prepare_cnn_input(path, size=(128, 128)):
    ela = convert_to_ela_image(path)
    if ela is None: return np.zeros((size[0], size[1], 3))
    return np.array(ela.resize(size)) / 255.0

def extract_benford(path):
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: return np.zeros(13)
        if img.shape[0] > 1000: img = cv2.resize(img, (0,0), fx=0.5, fy=0.5)
        img = np.float32(img)
        h, w = img.shape
        digits = []
        for i in range(0, h-8, 8):
            for j in range(0, w-8, 8):
                blk = cv2.dct(img[i:i+8, j:j+8])
                for v in blk.flatten()[1:]:
                    if abs(v) >= 1: digits.append(int(str(int(abs(v)))[0]))
        if not digits: return np.zeros(13)
        total = len(digits)
        obs = np.array([digits.count(d)/total for d in range(1,10)])
        exp = np.array([np.log10(1+1/d) for d in range(1,10)])
        chi = np.sum((obs-exp)**2/exp)*total
        ks = np.max(np.abs(np.cumsum(obs)-np.cumsum(exp)))
        mad = np.mean(np.abs(obs-exp))
        mse = np.mean((obs-exp)**2)
        return np.concatenate([obs, [chi/1000.0, ks, mad, mse]])
    except:
        return np.zeros(13)

## 4. Prepare All Data (Memory Intensive)
Load toàn bộ ảnh vào RAM để training nhanh.

In [ ]:
X_ela = []
X_benford = []
Y = []

print("Loading Data & Extracting Features...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    path = row['path']
    # 1. ELA Image
    X_ela.append(prepare_cnn_input(path))
    # 2. Benford Vector
    X_benford.append(extract_benford(path))
    # 3. Label
    Y.append(row['label'])

X_ela = np.array(X_ela)
X_benford = np.array(X_benford)
Y = np.array(Y)

print(f"X_ela: {X_ela.shape}")
print(f"X_benford: {X_benford.shape}")

# Split
X_ela_train, X_ela_test, X_ben_train, X_ben_test, Y_train, Y_test = train_test_split(
    X_ela, X_benford, Y, test_size=0.2, random_state=42, stratify=Y
)

## 5. Phase 1: Train CNN from Scratch

In [ ]:
def build_cnn():
    model = Sequential([
        Conv2D(32, (5,5), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()

print("Training CNN...")
history = cnn_model.fit(
    X_ela_train, Y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.1,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

cnn_model.save('cnn_ela_trained.h5')
print("CNN Model saved!")

## 6. Phase 2: Hybrid Training (SVM)

In [ ]:
# Get CNN Predictions (Probabilities)
cnn_prob_train = cnn_model.predict(X_ela_train)
cnn_prob_test = cnn_model.predict(X_ela_test)

# Combine with Benford
X_hybrid_train = np.hstack((cnn_prob_train, X_ben_train))
X_hybrid_test = np.hstack((cnn_prob_test, X_ben_test))

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_hybrid_train)
X_test_scaled = scaler.transform(X_hybrid_test)

# Train SVM
print("Training SVM...")
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, Y_train)

# Eval
preds = svm.predict(X_test_scaled)
probs = svm.predict_proba(X_test_scaled)[:, 1]
print("Hybrid Accuracy:", accuracy_score(Y_test, preds))

## 7. Visualization & Report

In [ ]:
def plot_metrics(y_true, y_pred, y_prob):
    plt.figure(figsize=(18, 5))
    
    # Confusion Matrix
    plt.subplot(1, 3, 1)
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    
    # ROC
    plt.subplot(1, 3, 2)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.plot(fpr, tpr, label=f'AUC={auc(fpr, tpr):.4f}')
    plt.plot([0,1],[0,1],'k--')
    plt.legend()
    plt.title('ROC Curve')
    
    # PR Curve
    plt.subplot(1, 3, 3)
    pre, rec, _ = precision_recall_curve(y_true, y_prob)
    plt.plot(rec, pre)
    plt.title('Precision-Recall Curve')
    
    plt.show()
    print(classification_report(y_true, y_pred, target_names=['Authentic', 'Forged']))

plot_metrics(Y_test, preds, probs)